# TechMentor-LLM — Colab Fine-Tuning with Unsloth

Fine-tune **Llama 3.2 3B Instruct** with **4-bit QLoRA** using [Unsloth](https://github.com/unslothai/unsloth) (faster training, lower VRAM on T4).

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**
2. Hugging Face access to [`meta-llama/Llama-3.2-3B-Instruct`](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
3. Optional: [Weights & Biases](https://wandb.ai) account

For the standard Hugging Face + PEFT trainer, use `colab_finetune_hf.ipynb` instead.

## 1. Configuration

In [ ]:
# @title Training settings
USE_WANDB = True

# Clone your repo or leave None if uploading manually to /content/TechMentor-LLM
REPO_URL = None  # e.g. "https://github.com/YOUR_USERNAME/TechMentor-LLM.git"

EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

## 2. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → T4 GPU"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")

## 3. Get project files

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/TechMentor-LLM")

if REPO_URL:
    if PROJECT_DIR.exists():
        !rm -rf {PROJECT_DIR}
    !git clone {REPO_URL} {PROJECT_DIR}
elif not PROJECT_DIR.exists():
    raise FileNotFoundError(
        "Set REPO_URL or upload the TechMentor-LLM folder to /content/TechMentor-LLM"
    )

%cd {PROJECT_DIR}
print("Project root:", PROJECT_DIR)

## 4. Install dependencies (Unsloth)

In [ ]:
%%capture
import torch

!pip install -q -r training/requirements.txt
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes datasets wandb

print("Unsloth training dependencies installed.")

## 5. Authenticate

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    login(userdata.get("HF_TOKEN"))
    print("Logged in to Hugging Face via Colab secret HF_TOKEN")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

if USE_WANDB:
    import wandb
    try:
        wandb.login(key=userdata.get("WANDB_API_KEY"))
        print("Logged in to W&B via Colab secret WANDB_API_KEY")
    except Exception:
        wandb.login()

## 6. Verify dataset

In [ ]:
import sys

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from training.config import EVAL_PATH, TRAIN_PATH

assert TRAIN_PATH.exists(), f"Missing {TRAIN_PATH} — run the data pipeline first"
assert EVAL_PATH.exists(), f"Missing {EVAL_PATH} — run the data pipeline first"

train_lines = TRAIN_PATH.read_text(encoding="utf-8").strip().splitlines()
eval_lines = EVAL_PATH.read_text(encoding="utf-8").strip().splitlines()
print(f"Train: {len(train_lines)} examples | Eval: {len(eval_lines)} examples")

## 7. Training setup

In [ ]:
import wandb
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

from training.config import (
    FINAL_ADAPTER_DIR,
    LOGS_DIR,
    MODEL_ID,
    OUTPUTS_DIR,
    WANDB_PROJECT,
    LoRAConfig,
    TrainingConfig,
    get_run_name,
)
from training.utils import (
    build_wandb_config,
    ensure_output_dirs,
    prepare_sft_datasets,
    save_adapter,
    save_training_artifacts,
)

lora = LoRAConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
training = TrainingConfig(
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_seq_length=MAX_SEQ_LENGTH,
)
run_name = get_run_name(lora, training)

ensure_output_dirs()
wandb_url = None

if USE_WANDB:
    run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        job_type="fine-tuning-unsloth",
        config=build_wandb_config(lora, training),
    )
    wandb_url = run.get_url()
    print("W&B run:", wandb_url)

print(f"Model: {MODEL_ID} | Run: {run_name}")

## 8. Train with Unsloth QLoRA

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=training.max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora.r,
    lora_alpha=lora.lora_alpha,
    lora_dropout=lora.lora_dropout,
    bias=lora.bias,
    target_modules=lora.target_modules,
    use_gradient_checkpointing="unsloth",
    random_state=training.seed,
)

train_dataset, eval_dataset = prepare_sft_datasets(tokenizer)
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

sft_config = SFTConfig(
    output_dir=str(OUTPUTS_DIR),
    num_train_epochs=training.epochs,
    per_device_train_batch_size=training.batch_size,
    per_device_eval_batch_size=training.batch_size,
    gradient_accumulation_steps=training.gradient_accumulation_steps,
    learning_rate=training.learning_rate,
    weight_decay=training.weight_decay,
    warmup_ratio=training.warmup_ratio,
    logging_steps=training.logging_steps,
    save_steps=training.save_steps,
    eval_steps=training.eval_steps,
    eval_strategy=training.evaluation_strategy,
    fp16=True,
    logging_dir=str(LOGS_DIR),
    report_to="wandb" if USE_WANDB else "none",
    seed=training.seed,
    max_seq_length=training.max_seq_length,
    dataset_text_field="text",
    packing=False,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,
)

trainer.train()
save_adapter(model, tokenizer, FINAL_ADAPTER_DIR)
save_training_artifacts(lora, training, wandb_url)

if USE_WANDB:
    wandb.finish()

print(f"Unsloth training complete → {FINAL_ADAPTER_DIR}")

## 9. Quick inference test

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

from training.prompts import TEST_PROMPTS, format_inference_prompt

assert FINAL_ADAPTER_DIR.exists(), "Train first — adapter not found"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
infer_model = PeftModel.from_pretrained(base, str(FINAL_ADAPTER_DIR))
infer_model.eval()


def generate(instruction: str, user_input: str, max_new_tokens: int = 256) -> str:
    prompt = format_inference_prompt(tokenizer, instruction, user_input)
    inputs = tokenizer(prompt, return_tensors="pt").to(infer_model.device)
    with torch.no_grad():
        outputs = infer_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


sample = TEST_PROMPTS[0]
print(f"[{sample['category']}] {sample['input']}")
print("-" * 60)
print(generate(sample["instruction"], sample["input"]))

## 10. Download adapter

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/techmentor_adapter_unsloth.zip"
shutil.make_archive("/content/techmentor_adapter_unsloth", "zip", FINAL_ADAPTER_DIR)
print(f"Zipped adapter: {zip_path}")
files.download(zip_path)

### Optional: save to Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

drive_dest = Path("/content/drive/MyDrive/techmentor_adapter_unsloth")
if drive_dest.exists():
    shutil.rmtree(drive_dest)
shutil.copytree(FINAL_ADAPTER_DIR, drive_dest)
print(f"Saved adapter to {drive_dest}")